# Gym Member Churn Analysis

## Project Overview
This notebook analyzes churn behavior among gym members using SQL (PostgreSQL) 
and Python. The goal is to identify key drivers of membership cancellation 
and derive actionable retention insights.

## Key Questions
1. What is the overall churn rate?
2. Does contract length affect churn?
3. Do inactive members churn more frequently?
4. Is churn influenced by age or gender?

## Tools
- **Python** — data loading and export
- **PostgreSQL** — SQL analysis via psycopg2
- **PowerBI** — dashboard visualization (separate file)

## Data Source
Dataset: [Gym Member Churn — kaggle](https://www.kaggle.com/datasets/hassaan2580/churn-prediction-gym-members-dataset)  
Records: 4,000 gym members | Features: 14 columns

## Data Loading

The dataset contains 4,000 gym member records sourced from kaggle, 
including behavioral and contractual features used to analyze churn patterns.

In [ ]:
# Load dataset into pandas DataFrame
import pandas as pd
df = pd.read_csv("gym_churn_us.csv")
df.head()


,gender,Near_Location,Partner,Promo_friends,Phone,Contract_period,Group_visits,Age,Avg_additional_charges_total,Month_to_end_contract,Lifetime,Avg_class_frequency_total,Avg_class_frequency_current_month,Churn
0,1,1,1,1,0,6,1,29,14.227470,5.0,3,0.020398,0.000000,0
1,0,1,0,0,1,12,1,31,113.202938,12.0,7,1.922936,1.910244,0
2,0,1,1,0,1,1,0,28,129.448479,1.0,2,1.859098,1.736502,0
3,0,1,1,1,1,12,1,33,62.669863,12.0,2,3.205633,3.357215,0
4,1,1,1,1,1,1,0,26,198.362265,1.0,3,1.113884,1.120078,0


## Overall Churn Rate

The overall churn rate represents the percentage of members who cancelled 
their membership. This serves as the baseline KPI for the entire analysis.

In [61]:
#  1. Overall Churn Rate
# SUM(churn)   = total members who cancelled (churn = 1)
# COUNT(*)     = total members in dataset
# * 100.0      = convert to percentage (forces decimal division)
# ROUND(...,2) = round to 2 decimal places
cursor.execute("""
    SELECT ROUND(SUM(churn) * 100.0 / COUNT(*), 2) AS overall_churn_rate
    FROM members
""")
result = cursor.fetchall()
print(f"Overall Churn Rate: {float(result[0][0])}%")

Overall Churn Rate: 26.53%


## Churn Rate by Contract Period

This query calculates the churn rate — the percentage of members who churned —
segmented by contract period. Grouping by `contract_period` reveals whether
longer or shorter contract durations correlate with higher customer attrition,
which is a key metric for retention analysis.

In [ ]:
# Aggregate churn rate (%) per contract period, rounded to 2 decimal places
cursor.execute("""
    SELECT ROUND(SUM(churn) * 100.0 / COUNT(*), 2), contract_period
    FROM members
    GROUP BY contract_period
    ORDER BY contract_period
""")

result = cursor.fetchall()  # Retrieve all rows from the query result

print(result)  # Inspect raw output before visualization or further processing

## Churn Rate by Contract Period — Results

The query returns a churn rate of **42.32%** for monthly contracts (1 month),
**12.48%** for semi-annual contracts (6 months), and **2.40%** for annual
contracts (12 months). This inverse relationship between contract duration and
churn rate suggests that members committing to longer contracts exhibit
significantly higher retention — a finding with direct implications for pricing
and subscription strategy.

In [17]:
# Unpack results into labeled variables for readability
churn_monthly, churn_semiannual, churn_annual = [float(row[0]) for row in result]

# Contract periods in months, corresponding to query output order
contract_labels = [1, 6, 12]
churn_rates     = [churn_monthly, churn_semiannual, churn_annual]

# Display a structured summary for quick reference
for label, rate in zip(contract_labels, churn_rates):
    print(f"Contract period: {label:>2} month(s) → Churn rate: {rate:.2f}%")

Contract period:  1 month(s) → Churn rate: 42.32%
Contract period:  6 month(s) → Churn rate: 12.48%
Contract period: 12 month(s) → Churn rate: 2.40%


## Activity Level by Churn Status

This query compares the average class attendance frequency between members who
churned and those who retained, using the current month's visit rate as a
behavioral proxy. The result demonstrates a substantial activity gap: retained
members attend on average **2.03 classes**, while churned members attended only
**1.04 classes** — suggesting that low engagement is a strong early indicator
of churn risk.

In [34]:
# Cast AVG result to numeric for ROUND compatibility (PostgreSQL-specific)
cursor.execute("""
    SELECT churn, ROUND(AVG(avg_class_frequency_current)::numeric, 2) AS avg_frequency
    FROM members
    GROUP BY churn
""")

result = cursor.fetchall()  # [(0, Decimal('2.03')), (1, Decimal('1.04'))]

# Unpack and label results for readability
for churn_status, avg_freq in result:
    label = "Churned" if churn_status == 1 else "Retained"
    print(f"{label}: avg. {float(avg_freq):.2f} classes/month")

Retained: avg. 2.03 classes/month
Churned: avg. 1.04 classes/month


## Age Distribution by Churn Status

This query calculates the average member age segmented by churn status to
assess whether younger members are more likely to cancel their membership.
The results indicate that churned members are on average **26.99 years old**,
compared to **29.98 years** among retained members — suggesting that younger
demographics exhibit higher attrition and may

In [38]:
# Compare average age between retained and churned members
cursor.execute("""
    SELECT churn, ROUND(AVG(age)::numeric, 2) AS avg_age
    FROM members
    GROUP BY churn
""")

result = cursor.fetchall()  # [(0, Decimal('29.98')), (1, Decimal('26.99'))]

# Unpack and label results for readability
for churn_status, avg_age in result:
    label = "Churned" if churn_status == 1 else "Retained"
    print(f"{label}: avg. age {float(avg_age):.2f} years")

Retained: avg. age 29.98 years
Churned: avg. age 26.99 years


## Churn Rate by Gender

This query calculates the churn rate separately for each gender to determine
whether attrition differs systematically between groups. The results show a
near-identical churn rate of **26.49%** (gender 0) and **26.56%** (gender 1),
indicating that gender has no meaningful predictive power for churn in this
dataset. With a difference of merely 0.07 percentage points, gender can be
treated as a **non-predictive feature** and deprioritized in subsequent
retention modeling or machine learning pipelines.

In [43]:
# Calculate churn rate (%) per gender group
cursor.execute("""
    SELECT ROUND(SUM(churn) * 100.0 / COUNT(*), 2), gender
    FROM members
    GROUP BY gender
    ORDER BY gender
""")

result = cursor.fetchall()  # [(Decimal('26.49'), '0'), (Decimal('26.56'), '1')]

# Unpack and label results for readability
for churn_rate, gender in result:
    label = "Female" if gender == '0' else "Male"  # Adjust mapping to your encoding
    print(f"{label}: churn rate {float(churn_rate):.2f}%")

Female: churn rate 26.49%
Male: churn rate 26.56%


## Exporting Churn Rate by Contract Period to CSV

This query retrieves the previously established churn rates per contract period
and loads the result directly into a pandas DataFrame for structured handling.
The data is exported as a CSV file to enable reproducible visualization and
further analysis outside the notebook environment.

In [58]:
import pandas as pd

# Query churn rate per contract period with column alias for clean DataFrame mapping
cursor.execute("""
    SELECT contract_period, ROUND(SUM(churn) * 1.0 / COUNT(*), 6) AS churn_rate
    FROM members
    GROUP BY contract_period
    ORDER BY contract_period
""")

# Load query result into a DataFrame with explicit column names
df_contract = pd.DataFrame(cursor.fetchall(), columns=["contract_period", "churn_rate"])

# Persist to CSV for downstream visualization or reporting
df_contract.to_csv("churn_by_contract.csv", index=False)

print(df_contract)  # Confirm structure before proceeding to visualization

   contract_period churn_rate
0                1   0.423199
1                6   0.124850
2               12   0.023958


## Exporting Average Activity Level by Churn Status to CSV

This query retrieves the average class attendance frequency segmented by churn
status and loads the result into a pandas DataFrame for structured handling.
The data is exported as a CSV file to enable reproducible visualization and
further analysis outside the notebook environment.

In [53]:
# Query average class frequency segmented by churn status
cursor.execute("""
    SELECT churn, ROUND(AVG(avg_class_frequency_current)::numeric, 2) AS avg_frequency
    FROM members
    GROUP BY churn
    ORDER BY churn
""")

# Load result into DataFrame with explicit column names
df_activity = pd.DataFrame(cursor.fetchall(), columns=["churn", "avg_frequency"])

# Persist to CSV for downstream visualization or reporting
df_activity.to_csv("churn_by_activity.csv", index=False)

print(df_activity)  # Confirm structure before proceeding to visualization

   churn avg_frequency
0      0          2.03
1      1          1.04


## Exporting Average Age by Churn Status to CSV

This query retrieves the average member age segmented by churn status and loads
the result into a pandas DataFrame for structured handling. The data is exported
as a CSV file to enable reproducible visualization and further analysis outside
the notebook environment.

In [47]:
# Query average age segmented by churn status
cursor.execute("""
    SELECT churn, ROUND(AVG(age)::numeric, 2) AS avg_age
    FROM members
    GROUP BY churn
    ORDER BY churn
""")

# Load result into DataFrame with explicit column names
df_age = pd.DataFrame(cursor.fetchall(), columns=["churn", "avg_age"])

# Persist to CSV for downstream visualization or reporting
df_age.to_csv("churn_by_age.csv", index=False)

print(df_age)  # Confirm structure before proceeding to visualization

   churn avg_age
0      0   29.98
1      1   26.99


## Exporting Churn Rate by Gender to CSV

This query retrieves the churn rate segmented by gender and loads the result
into a pandas DataFrame for structured handling. Despite the negligible
difference between groups, the data is exported as a CSV file to maintain
consistency across the analysis pipeline and to document gender as a
non-predictive feature.

In [59]:
# Query churn rate segmented by gender
cursor.execute("""
    SELECT gender, ROUND(SUM(churn) * 1.0/ COUNT(*), 6) AS churn_rate
    FROM members
    GROUP BY gender
    ORDER BY gender
""")

# Load result into DataFrame with explicit column names
df_gender = pd.DataFrame(cursor.fetchall(), columns=["gender", "churn_rate"])

# Persist to CSV for downstream visualization or reporting
df_gender.to_csv("churn_by_gender.csv", index=False)

print(df_gender)  # Confirm structure before proceeding to visualization

  gender churn_rate
0      0   0.264931
1      1   0.265556


## Export KPIs for PowerBI

Key performance indicators are exported as a CSV for dashboard visualization in PowerBI.
Values are stored as decimals (e.g. 0.2653 = 26.53%) for compatibility with PowerBI percentage formatting.

In [62]:
import pandas as pd

# Define KPI metrics derived from SQL analysis
df_kpi = pd.DataFrame({
    "metric": ["Total Members", "Overall Churn Rate", "Monthly Contract Churn", "Annual Contract Churn"],
    "value": [4000, 0.2653, 0.4232, 0.0240]  # Decimal format for PowerBI percentage rendering
})

# Export to CSV for PowerBI import
df_kpi.to_csv("churn_kpi.csv", index=False)
print("KPI export complete")

KPI export complete
